# Retrain the parser on reworded requests (Kaggle GPU T4 x2, one session)

The 5-fold parser (Qwen3-1.7B + LoRA) fits SimuHome's template phrasing: on meaning-preserving rewordings it
missed most QT4-1 contradictions. This notebook retrains it on the original requests **plus** rewordings of
the same training requests, and tests it on rewordings by a **different** model it never trained on.

| Stage | What | Time (T4 x2, rough) |
|---|---|---|
| 1 | Reword all 300 requests twice: Llama-3.1-8B (training) and Mistral-Nemo-12B (test); each rewording is checked for meaning | ~1 h |
| 2 | Retrain the 5 fold parsers: fold k = its original training requests + their Llama rewordings (never its own test requests) | ~2-2.5 h |
| 3 | Parse with OLD and NEW parsers: original requests, Mistral rewordings (+ Qwen rewordings if attached), each fold's held-out requests only | ~1 h |
| 4 | Scores (original vs reworded, old vs new) + how many gate decisions on the ORIGINAL requests change | minutes |

**Attach:** `parser_v3_cv_outputs` (the zip or dataset with `parse_cache_v3_cv.json` and the 5 adapters
`qwen3_ir_parser_adapter_v3_fold0..4`). Optional: the `rewordings.jsonl` from notebook 03 (Qwen2.5-14B rewordings).
**Settings:** GPU T4 x2, Internet ON. Every stage resumes: if the session stops, run all cells again.
**Output:** `reworded_parser_results.zip` (report, rewordings, parse caches) and `reworded_parser_adapters.zip`.


## 1. Setup


In [ ]:
!pip install -q z3-solver
!pip install -q -U transformers peft accelerate trl datasets
!pip uninstall -y -q torchao
import os, sys, glob, json, re, time, random, hashlib, shutil, gc, subprocess, zipfile, urllib.request, collections, copy
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
os.chdir(WORK)
if not os.path.isdir('SimuHome'):
    !git clone --depth 1 https://github.com/holi-lab/SimuHome.git
BENCH = 'SimuHome/data/benchmark'
assert len(glob.glob(f'{BENCH}/qt4-*.json')) == 300, 'SimuHome clone incomplete'
sys.path.insert(0, '.')


In [ ]:
import os; os.makedirs('irlib', exist_ok=True); open('irlib/__init__.py', 'a').close()


In [ ]:
%%writefile irlib/ir_schema.py
"""Typed intermediate representation (IR v3) for SimuHome QT4 scheduling requests.

An IR describes one `schedule_workflow`-shaped user request as a set of goals.
Each goal is a set of target device assertions gated by one or more *time
readings* (ways the natural-language query says when the action happens).
A feasible request has one consistent reading per goal; an infeasible one states
two readings that resolve to different times for the same action (SimuHome's
"Contradiction Blindness" failure mode).

Changes from v2 (approach 1):
  - Anchor kind "goal": a reading measured from an EARLIER goal of the same
    request ("8 minutes after the previous action"). v2 had no such anchor and
    labelled these readings as "now"-anchored, which is harmless for a pure
    contradiction check but wrong for checking the agent's start_time.
  - Relation "by": a deadline (the state must hold no later than t). SimuHome
    marks these goals with tolerance_ticks == 0 and relation "before" (e.g. QT4-3
    "start the dryer now ... pause it when the washer finishes": the dryer only
    has to be running before the washer ends). v2 treated them as a point.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Literal, Optional

Relation = Literal["after", "before", "at", "by"]
AnchorKind = Literal["now", "device_event", "goal"]

IR_VERSION = 3


@dataclass
class Anchor:
    """What a time offset is measured from."""

    kind: AnchorKind = "now"
    room_id: Optional[str] = None
    device_id: Optional[str] = None
    device_type: Optional[str] = None
    goal_ref: Optional[int] = None  # kind == "goal": index of the earlier goal

    def to_dict(self) -> dict[str, Any]:
        return {
            "kind": self.kind,
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "goal_ref": self.goal_ref,
        }


@dataclass
class TimeReading:
    """One way the query expresses *when* an action should occur.

    All times are minutes since the episode's `base_time` (the moment the user
    spoke). A second reading on the same goal that disagrees with the first is
    what makes a request infeasible.
    """

    relation: Relation
    anchor: Anchor
    offset_minutes: Optional[float] = None  # relative reading
    absolute_time: Optional[str] = None  # "HH:MM[:SS]", clock-time reading
    # device_event anchors only: when the anchor device finishes (minutes since
    # base_time). Never in the training target; filled from live simulator state.
    resolved_at_minutes: Optional[float] = None
    tolerance_minutes: float = 0.0

    def to_dict(self) -> dict[str, Any]:
        return {
            "relation": self.relation,
            "anchor": self.anchor.to_dict(),
            "offset_minutes": self.offset_minutes,
            "absolute_time": self.absolute_time,
            "resolved_at_minutes": self.resolved_at_minutes,
            "tolerance_minutes": self.tolerance_minutes,
        }


@dataclass
class Assertion:
    attribute: str
    value: Any
    value_type: Optional[str] = None
    description: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "attribute": self.attribute,
            "value": self.value,
            "value_type": self.value_type,
            "description": self.description,
        }


@dataclass
class Target:
    room_id: str
    device_id: str
    device_type: Optional[str]
    asserts: list[Assertion] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "asserts": [a.to_dict() for a in self.asserts],
        }


@dataclass
class Goal:
    """One scheduled effect: a set of device assertions gated by time reading(s)."""

    goal_id: int
    readings: list[TimeReading]
    targets: list[Target]
    # Set when the conflict is a duration/resource issue rather than a stated-time
    # mismatch (e.g. "finish the washer exactly when the dishwasher finishes" but the
    # washer cycle is longer). Not checkable without a device duration model.
    requires_duration_model: bool = False

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal_id": self.goal_id,
            "readings": [r.to_dict() for r in self.readings],
            "targets": [t.to_dict() for t in self.targets],
            "requires_duration_model": self.requires_duration_model,
        }


@dataclass
class IntentIR:
    query: str
    query_type: str
    case: Literal["feasible", "infeasible", "unknown"]
    base_time: str  # "YYYY-MM-DD HH:MM:SS"
    goals: list[Goal] = field(default_factory=list)
    conflict_type: Optional[str] = None  # ground-truth label, training-time only

    def to_dict(self) -> dict[str, Any]:
        return {
            "query": self.query,
            "query_type": self.query_type,
            "case": self.case,
            "base_time": self.base_time,
            "goals": [g.to_dict() for g in self.goals],
            "conflict_type": self.conflict_type,
        }


def ir_from_dict(d: dict) -> IntentIR:
    """Inverse of IntentIR.to_dict() (used for the debug ground-truth IRs)."""
    goals = []
    for g in d["goals"]:
        readings = [
            TimeReading(anchor=Anchor(**r["anchor"]), **{k: v for k, v in r.items() if k != "anchor"})
            for r in g["readings"]
        ]
        targets = [
            Target(
                room_id=t["room_id"],
                device_id=t["device_id"],
                device_type=t.get("device_type"),
                asserts=[Assertion(**a) for a in t.get("asserts", [])],
            )
            for t in g["targets"]
        ]
        goals.append(
            Goal(
                goal_id=g["goal_id"],
                readings=readings,
                targets=targets,
                requires_duration_model=g.get("requires_duration_model", False),
            )
        )
    return IntentIR(
        query=d["query"],
        query_type=d["query_type"],
        case=d["case"],
        base_time=d["base_time"],
        goals=goals,
        conflict_type=d.get("conflict_type"),
    )


# ---------------------------------------------------------------------------
# Compact wire format (what the SLM is trained to emit)
# ---------------------------------------------------------------------------
# Keeps only what the query text determines: per goal, the time readings and the
# targets. Z3 decides feasibility; the model never says it. v3 adds `goal_ref`
# for goal anchors and the "by" relation; everything else is unchanged from v2,
# so a v2 parser's output still loads (it just never produces goal anchors).

DEFAULT_TOLERANCE_MINUTES = 0.5


def compact_ir_dict(ir_dict: dict, *, keep_resolved: bool = False) -> dict:
    """Full IR dict -> compact dict. Clock times are cut to HH:MM (the query
    states minutes, never the simulator's seconds)."""
    goals = []
    for g in ir_dict["goals"]:
        readings = []
        for r in g["readings"]:
            anchor = {"kind": r["anchor"]["kind"]}
            if r["anchor"].get("device_id"):
                anchor["device_id"] = r["anchor"]["device_id"]
            if r["anchor"].get("room_id"):
                anchor["room_id"] = r["anchor"]["room_id"]
            if r["anchor"].get("goal_ref") is not None:
                anchor["goal_ref"] = r["anchor"]["goal_ref"]
            out = {"relation": r["relation"], "anchor": anchor}
            if r.get("offset_minutes") is not None:
                out["offset_minutes"] = r["offset_minutes"]
            if r.get("absolute_time") is not None:
                out["absolute_time"] = ":".join(r["absolute_time"].split(":")[:2])
            if keep_resolved and r.get("resolved_at_minutes") is not None:
                out["resolved_at_minutes"] = r["resolved_at_minutes"]
            readings.append(out)
        targets = [
            {
                "device_id": t["device_id"],
                "asserts": [{"attribute": a["attribute"], "value": a["value"]} for a in t.get("asserts", [])],
            }
            for t in g["targets"]
        ]
        goals.append({"readings": readings, "targets": targets})
    return {"goals": goals}


def ir_from_compact(compact: dict, query: str, base_time: str) -> IntentIR:
    """Compact dict (+ the live query and the episode base_time) -> IntentIR."""
    goals = []
    for i, g in enumerate(compact["goals"]):
        readings = []
        for r in g["readings"]:
            a = r.get("anchor") or {}
            readings.append(
                TimeReading(
                    relation=r.get("relation", "after"),
                    anchor=Anchor(
                        kind=a.get("kind", "now"),
                        room_id=a.get("room_id"),
                        device_id=a.get("device_id"),
                        goal_ref=a.get("goal_ref"),
                    ),
                    offset_minutes=r.get("offset_minutes"),
                    absolute_time=r.get("absolute_time"),
                    resolved_at_minutes=r.get("resolved_at_minutes"),
                    tolerance_minutes=r.get("tolerance_minutes", DEFAULT_TOLERANCE_MINUTES),
                )
            )
        targets = [
            Target(
                room_id=t.get("room_id", ""),
                device_id=t["device_id"],
                device_type=None,
                asserts=[Assertion(attribute=a["attribute"], value=a["value"]) for a in t.get("asserts", [])],
            )
            for t in g.get("targets", [])
        ]
        goals.append(Goal(goal_id=i, readings=readings, targets=targets))
    return IntentIR(query=query, query_type="qt4", case="unknown", base_time=base_time, goals=goals)


def uses_goal_anchors(compact: dict) -> bool:
    """True if a compact IR contains any v3 goal anchor (i.e. came from a v3 parser
    or from v3 labels). Used to decide whether start_time checks are trustworthy."""
    return any(
        (r.get("anchor") or {}).get("kind") == "goal"
        for g in compact.get("goals", [])
        for r in g.get("readings", [])
    )


In [ ]:
%%writefile irlib/constraint_encoder.py
"""Component 2: IR -> Z3 constraints over real-valued time; SAT/UNSAT and time windows.

Time is one Real variable per goal: minutes since the episode's `base_time` (the
moment the user spoke, "now" = 0). Each TimeReading adds one constraint.

Changes from approach 1 (which solved every goal separately):
  - All goals are encoded in ONE solver, so readings can refer to each other:
    goal anchors ("8 min after the previous action") and a device anchor shared
    by several goals ("10 and 20 min after the dishwasher finishes" must agree on
    when the dishwasher finishes).
  - "by" readings are deadlines (t <= target + tol), not points.
  - When the request is consistent, `goal_windows` returns the interval [lo, hi]
    of start times each goal allows. The gate checks the agent's
    `schedule_workflow(start_time=...)` against these windows. This is the part a
    one-line Python comparison cannot do once goals chain off each other.

CPU-only; no GPU, simulator or SLM needed to run or test this module.
"""

from __future__ import annotations

import functools
import threading
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional

import z3

from .ir_schema import Goal, IntentIR, TimeReading

# Z3's default context is not thread-safe: two episodes checking at the same time
# (the parallel runner) crash it with "unreachable" / "invalid argument". Every
# public entry point holds this lock; a check takes milliseconds, so it costs nothing.
_Z3_LOCK = threading.RLock()


def _z3_serialized(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with _Z3_LOCK:
            return fn(*args, **kwargs)
    return wrapper


def _parse_hms_to_minutes_of_day(hms: str) -> float:
    parts = [int(x) for x in hms.split(":")]
    h, m = parts[0], parts[1]
    s = parts[2] if len(parts) > 2 else 0
    return h * 60.0 + m + s / 60.0


def _is_minute_granular(hms: str) -> bool:
    return len(hms.split(":")) < 3


def _base_time_minutes_of_day(base_time: str) -> float:
    return _parse_hms_to_minutes_of_day(base_time.split(" ", 1)[1])


def minutes_to_clock(base_time: str, minutes: float) -> str:
    """Minutes since base_time -> "HH:MM:SS" wall-clock string."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (base + timedelta(minutes=float(minutes))).strftime("%H:%M:%S")


def clock_to_minutes(base_time: str, timestamp: str) -> float:
    """"YYYY-MM-DD HH:MM:SS" -> minutes since base_time."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S") - base).total_seconds() / 60.0


def reading_to_absolute_minutes(reading: TimeReading, base_time: str) -> Optional[float]:
    """Minutes since base_time for a self-contained clock-time reading, else None."""
    if reading.absolute_time is not None and reading.anchor.kind == "now":
        # Clock times are same-day in SimuHome QT4 episodes; no midnight wrap handling.
        return _parse_hms_to_minutes_of_day(reading.absolute_time) - _base_time_minutes_of_day(base_time)
    return None


@dataclass
class GoalCheckResult:
    goal_id: int
    sat: bool
    reason: Optional[str] = None  # populated on UNSAT (or "skipped: ...")
    unsat_core: Optional[list[str]] = None
    resolved_minutes: Optional[float] = None  # one consistent time, on SAT


@dataclass
class GoalWindow:
    """Start times (minutes since base_time) a goal allows. None = unbounded."""

    goal_id: int
    lo: Optional[float]
    hi: Optional[float]
    deadline: bool = False  # only an upper bound was stated ("by")

    def contains(self, minutes: float, slack: float = 0.0) -> bool:
        if self.lo is not None and minutes < self.lo - slack:
            return False
        if self.hi is not None and minutes > self.hi + slack:
            return False
        return True


@dataclass
class FeasibilityResult:
    feasible: bool
    goal_results: list[GoalCheckResult]
    base_time: Optional[str] = None
    core_descriptions: list[str] = field(default_factory=list)

    def explanation(self) -> str:
        bad = [g for g in self.goal_results if not g.sat]
        if not bad:
            return "All scheduling constraints are consistent."
        lines = [f"Goal {g.goal_id}: {g.reason}" for g in bad]
        return "Infeasible schedule - " + "; ".join(lines)


def _describe_reading(reading: TimeReading, base_time: Optional[str], value: Optional[float]) -> str:
    when = f" (= {minutes_to_clock(base_time, value)})" if (base_time and value is not None) else ""
    if reading.absolute_time is not None:
        return f"stated clock time {reading.absolute_time}"
    if reading.offset_minutes is None:
        return "unspecified time reading"
    if reading.anchor.kind == "now":
        anchor_desc = "now"
    elif reading.anchor.kind == "goal":
        anchor_desc = f"goal {reading.anchor.goal_ref}"
    else:
        anchor_desc = f"{reading.anchor.device_id} finishing"
    rel = "before" if reading.relation == "by" else reading.relation
    prefix = "no later than " if reading.relation == "by" else ""
    return f"{prefix}{reading.offset_minutes:g} min {rel} {anchor_desc}{when}"


class _Encoding:
    """Builds the joint constraint system once; reused for the check and the windows."""

    def __init__(self, ir: IntentIR):
        self.ir = ir
        self.t = {g.goal_id: z3.Real(f"t_goal_{g.goal_id}") for g in ir.goals}
        self.anchors: dict[str, z3.ArithRef] = {}
        self.pins: list[tuple[str, z3.BoolRef]] = []  # known anchor times (not "readings")
        self.readings: list[tuple[str, z3.BoolRef, int, TimeReading, Optional[float]]] = []
        self.skipped: dict[int, str] = {}
        for goal in ir.goals:
            self._encode_goal(goal)

    def _anchor_var(self, device_id: str, resolved: Optional[float], label: str) -> z3.ArithRef:
        if device_id not in self.anchors:
            self.anchors[device_id] = z3.Real(f"anchor_{device_id}")
        var = self.anchors[device_id]
        if resolved is not None:
            self.pins.append((f"{label}_anchor_known", var == z3.RealVal(resolved)))
        return var

    def _encode_goal(self, goal: Goal) -> None:
        if goal.requires_duration_model:
            self.skipped[goal.goal_id] = "skipped: requires a device duration model (not checked)"
            return
        t = self.t[goal.goal_id]
        for i, reading in enumerate(goal.readings):
            label = f"g{goal.goal_id}_r{i}"
            extra = 1.0 if (reading.absolute_time and _is_minute_granular(reading.absolute_time)) else 0.0
            # An "HH:MM" clock time is only known to the minute, so widen by one minute.
            tol = z3.RealVal(reading.tolerance_minutes + extra)
            sign = -1 if reading.relation in ("before", "by") else 1
            nominal: Optional[float] = None  # the reading's own time, when known, for explanations

            if reading.anchor.kind == "device_event":
                anchor_var = self._anchor_var(reading.anchor.device_id or "?", reading.resolved_at_minutes, label)
                if reading.offset_minutes is None:
                    continue
                offset = 0.0 if reading.relation == "at" else sign * reading.offset_minutes
                target = anchor_var + z3.RealVal(offset)
                if reading.resolved_at_minutes is not None:
                    nominal = reading.resolved_at_minutes + offset
            elif reading.anchor.kind == "goal":
                ref = reading.anchor.goal_ref
                if ref is None or ref not in self.t or ref == goal.goal_id or reading.offset_minutes is None:
                    continue  # malformed reference (e.g. a parser slip): no constraint rather than a wrong one
                target = self.t[ref] + z3.RealVal(sign * reading.offset_minutes)
            else:
                resolved = reading_to_absolute_minutes(reading, self.ir.base_time)
                if resolved is not None:
                    target = z3.RealVal(resolved)
                    nominal = resolved
                elif reading.offset_minutes is not None:
                    target = z3.RealVal(sign * reading.offset_minutes)
                    nominal = sign * reading.offset_minutes
                else:
                    continue

            if reading.relation == "by":
                constraint = t <= target + tol
            else:
                constraint = z3.And(t >= target - tol, t <= target + tol)
            self.readings.append((label, constraint, goal.goal_id, reading, nominal))

    def solver(self, track: bool) -> z3.Solver:
        s = z3.Solver()
        for label, c in self.pins:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        for label, c, *_ in self.readings:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        return s


@_z3_serialized
def check_feasibility(ir: IntentIR) -> FeasibilityResult:
    enc = _Encoding(ir)
    solver = enc.solver(track=True)
    if solver.check() == z3.sat:
        model = solver.model()
        results = []
        for g in ir.goals:
            if g.goal_id in enc.skipped:
                results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped[g.goal_id]))
                continue
            val = model.eval(enc.t[g.goal_id], model_completion=True)
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, resolved_minutes=float(val.as_fraction())))
        return FeasibilityResult(feasible=True, goal_results=results, base_time=ir.base_time)

    core = {str(c) for c in solver.unsat_core()}
    by_label = {label: (gid, r, nominal) for label, _, gid, r, nominal in enc.readings}
    bad_goals: dict[int, list[str]] = {}
    descriptions = []
    # "*_anchor_known" pins only fix a symbolic anchor to a runtime-known value;
    # they are not a statement in the request, so they are not named.
    for label in sorted(core):
        if label in by_label:
            gid, r, nominal = by_label[label]
            d = _describe_reading(r, ir.base_time, nominal)
            bad_goals.setdefault(gid, []).append(d)
            descriptions.append(f"goal {gid}: {d}")
    results = []
    for g in ir.goals:
        if g.goal_id in bad_goals:
            results.append(
                GoalCheckResult(
                    goal_id=g.goal_id,
                    sat=False,
                    reason=" vs. ".join(bad_goals[g.goal_id]),
                    unsat_core=sorted(core),
                )
            )
        else:
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped.get(g.goal_id)))
    if not bad_goals:  # core made only of pins (should not happen); still report UNSAT honestly
        results[0] = GoalCheckResult(goal_id=results[0].goal_id, sat=False, reason="unsatisfiable", unsat_core=sorted(core))
    return FeasibilityResult(feasible=False, goal_results=results, base_time=ir.base_time, core_descriptions=descriptions)


def _bound(enc: _Encoding, var: z3.ArithRef, maximize: bool) -> Optional[float]:
    opt = z3.Optimize()
    for _, c in enc.pins:
        opt.add(c)
    for _, c, *_ in enc.readings:
        opt.add(c)
    h = opt.maximize(var) if maximize else opt.minimize(var)
    if opt.check() != z3.sat:
        return None
    value = opt.upper(h) if maximize else opt.lower(h)
    # Z3 hands back an integer or rational numeral when the bound is finite, and a
    # non-numeral term (oo, epsilon) when the goal is unbounded in that direction.
    if z3.is_int_value(value):
        return float(value.as_long())
    if z3.is_rational_value(value):
        return float(value.as_fraction())
    return None


@_z3_serialized
def goal_windows(ir: IntentIR) -> dict[int, GoalWindow]:
    """For a consistent IR: the [lo, hi] start-time interval of every goal.

    Goals whose readings are all unresolved (e.g. the anchor device is idle, so its
    finish time is unknown) come back unbounded, and the gate does not check them.
    """
    enc = _Encoding(ir)
    if enc.solver(track=False).check() != z3.sat:
        return {}
    windows = {}
    for g in ir.goals:
        if g.goal_id in enc.skipped:
            windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=None, hi=None)
            continue
        t = enc.t[g.goal_id]
        lo, hi = _bound(enc, t, maximize=False), _bound(enc, t, maximize=True)
        deadline = all(r.relation == "by" for r in g.readings) and bool(g.readings)
        windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=lo, hi=hi, deadline=deadline)
    return windows


In [ ]:
%%writefile irlib/extract_ir_from_episode.py
"""Component: deterministic (query, IR) pair extraction from SimuHome episodes.

No LLM calls, no manual labeling. SimuHome benchmark/generated episodes
already carry, in `eval.goals` (ground-truth required behavior) and
`temporal_conflict` (ground-truth contradiction, infeasible episodes only),
every field needed to derive the IR mechanically. See handoff Sec 5.3.

Two IRs are produced per episode:
  - `ir_for_training`: what the SLM must learn to produce from the query
    text alone. Device-anchored readings never carry a numeric resolved
    time here, because the query text does not state one — the live
    simulator state does, and only the runtime agent can fetch it.
  - `debug_ground_truth`: the same IR with anchor times filled in from the
    simulator-computed fields, for exercising the Z3 encoder end-to-end
    without needing a live simulator.

IR v3 (approach 2) differs from v2 in two labelling rules, both read off
`eval.goals[].when`:
  - a goal with no device anchor but `reference_minutes > 0` is chained to the
    earlier goal whose `at_minutes` equals that reference ("8 minutes after the
    previous action") -> Anchor(kind="goal", goal_ref=j);
  - `relation == "before"` with `tolerance_ticks == 0` is a deadline (the state
    must hold by then, e.g. QT4-3 "start the dryer now") -> relation "by".
"""

from __future__ import annotations

import glob
import json
import os
from typing import Any, Optional

from .ir_schema import Anchor, Assertion, Goal, IntentIR, Target, TimeReading, compact_ir_dict

# Conflict types where the contradiction is a stated-time mismatch (checkable
# from two TimeReadings alone). See handoff Sec 6: duration/resource
# feasibility conflicts (e.g. "finish washer exactly when dishwasher ends"
# but the washer cycle itself is longer) need a device duration model the
# Z3 encoder does not have yet — flagged, not silently mis-scored.
_DURATION_MODEL_CONFLICT_TYPES = {"completion_vs_pause"}


def _tolerance_minutes(when: dict[str, Any], tick_interval: float) -> float:
    return float(when.get("tolerance_ticks", 0)) * tick_interval / 60.0


def _chained_goal_ref(
    goal_id: int, when: dict[str, Any], raw_goals: list[dict[str, Any]], *, any_direction: bool = False
) -> Optional[int]:
    """Index of the goal this one is measured from, or None.

    SimuHome writes "8 minutes after the previous action" as relation "after",
    offset 8, with `reference_minutes` = the previous goal's `at_minutes` and no
    device anchor; the latest earlier goal with that time is the reference.
    Deadline goals (QT4-3, `any_direction=True`) reference the OTHER goal's time
    ("the dryer must be running 1 min before it is paused"), which can be a later goal.
    """
    ref = when.get("reference_minutes")
    if not ref:
        return None
    for j in range(goal_id - 1, -1, -1):
        if raw_goals[j]["when"].get("at_minutes") == ref:
            return j
    if any_direction:
        for j in range(goal_id + 1, len(raw_goals)):
            if raw_goals[j]["when"].get("at_minutes") == ref:
                return j
    return None


def _device_anchor(raw_anchor: dict[str, Any]) -> Anchor:
    return Anchor(
        kind="device_event",
        room_id=raw_anchor.get("room_id"),
        device_id=raw_anchor.get("device_id"),
        device_type=raw_anchor.get("device_type"),
    )


def _goal_from_eval_goal(
    goal_id: int,
    raw_goal: dict[str, Any],
    tick_interval: float,
    raw_goals: Optional[list[dict[str, Any]]] = None,
    for_training: bool = True,
) -> Goal:
    when = raw_goal["when"]
    raw_anchor = raw_goal.get("anchor") or {}

    relation = when.get("relation", "after")
    deadline = relation == "before" and int(when.get("tolerance_ticks", 0)) == 0
    if deadline:
        relation = "by"  # deadline: the state must hold no later than anchor - offset

    goal_ref = _chained_goal_ref(goal_id, when, raw_goals or [], any_direction=deadline)
    if deadline and goal_ref is not None:
        # Measured from the other goal, even when eval.goals labels a device anchor:
        # its reference_minutes is that goal's time, not the device's finish.
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    elif raw_anchor.get("device_id"):
        anchor = _device_anchor(raw_anchor)
    elif goal_ref is not None:
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    else:
        anchor = Anchor(kind="now")

    reading = TimeReading(
        relation=relation,
        anchor=anchor,
        offset_minutes=float(when.get("offset_minutes", 0.0)),
        tolerance_minutes=_tolerance_minutes(when, tick_interval),
    )
    if anchor.kind == "device_event" and not for_training and when.get("reference_tick") is not None:
        # Debug IR only: when the anchor device finishes, as the runtime reads it from
        # the simulator (CountdownTime). The training IR never contains it. Ticks, not
        # reference_minutes, which SimuHome rounds to whole minutes.
        reading.resolved_at_minutes = float(when["reference_tick"]) * tick_interval / 60.0

    targets = [
        Target(
            room_id=t["room_id"],
            device_id=t["device_id"],
            device_type=t.get("device_type"),
            asserts=[
                Assertion(
                    attribute=a["attribute"],
                    value=a["value"],
                    value_type=a.get("value_type"),
                    description=a.get("description"),
                )
                for a in t.get("asserts", [])
            ],
        )
        for t in raw_goal.get("targets", [])
    ]

    return Goal(goal_id=goal_id, readings=[reading], targets=targets)


def _inject_conflicting_reading(
    goal: Goal, temporal_conflict: dict[str, Any], for_training: bool, raw_anchor: Optional[dict[str, Any]] = None
) -> Goal:
    """Add the second, contradicting TimeReading to the primary goal.

    Mutates and returns `goal`. If the conflict is duration/resource-based
    (no comparable second time reading exists), flags the goal instead.
    """
    conflict_type = temporal_conflict.get("type")

    if conflict_type in _DURATION_MODEL_CONFLICT_TYPES or "conflict_at" not in temporal_conflict:
        goal.requires_duration_model = True
        return goal

    anchor_end_at = temporal_conflict.get("anchor_end_at")
    expected_at = temporal_conflict.get("expected_at")
    conflict_time = temporal_conflict.get("conflict_time")
    relation = temporal_conflict.get("relation", "after")

    if anchor_end_at is not None:
        # Dependency-anchored primary reading (e.g. "20 min after dishwasher finishes").
        base_offset = temporal_conflict.get(
            "offset_minutes", temporal_conflict.get("delay_minutes", 0.0)
        )
        offset = float(base_offset)
        # The conflict is stated against the anchor DEVICE finishing, so use the
        # device from eval.goals even if v3 relabelled the goal as goal-anchored.
        anchor = _device_anchor(raw_anchor) if (raw_anchor or {}).get("device_id") else goal.readings[0].anchor
        primary_reading = TimeReading(
            relation=relation,
            anchor=anchor,
            offset_minutes=offset,
            tolerance_minutes=goal.readings[0].tolerance_minutes,
            # resolved_at_minutes here is the ANCHOR EVENT's own resolved
            # time (e.g. when the dishwasher finishes), not the target time
            # — the Z3 encoder derives target = anchor +/- offset itself.
            resolved_at_minutes=None if for_training else float(anchor_end_at),
        )
        goal.readings[0] = primary_reading
    elif expected_at is not None:
        # Now-anchored primary reading (e.g. "13 minutes from now").
        goal.readings[0].offset_minutes = float(expected_at)
        goal.readings[0].anchor = Anchor(kind="now")

    if conflict_time is not None:
        goal.readings.append(
            TimeReading(
                relation="at",
                anchor=Anchor(kind="now"),
                absolute_time=conflict_time.split(" ")[-1] if " " in conflict_time else conflict_time,
                tolerance_minutes=goal.readings[0].tolerance_minutes,
            )
        )

    return goal


def extract_episode(episode: dict[str, Any], *, for_training: bool = True) -> IntentIR:
    """Build an IntentIR from one raw SimuHome benchmark/generated episode dict."""
    meta = episode["meta"]
    tick_interval = float(episode["initial_home_config"]["tick_interval"])
    base_time = episode["initial_home_config"]["base_time"]
    case = meta.get("case", "unknown")

    raw_goals = episode["eval"]["goals"]
    goals = [
        _goal_from_eval_goal(i, rg, tick_interval, raw_goals, for_training) for i, rg in enumerate(raw_goals)
    ]

    conflict_type = None
    if case == "infeasible" and "temporal_conflict" in episode:
        temporal_conflict = episode["temporal_conflict"]
        conflict_type = temporal_conflict.get("type")
        first = goals[0].readings[0] if goals and goals[0].readings else None
        if (
            first is not None
            and first.relation == "by"
            and first.anchor.kind == "goal"
            and conflict_type not in _DURATION_MODEL_CONFLICT_TYPES
            and temporal_conflict.get("conflict_time")
        ):
            # QT4-3: every goal keeps exactly the reading its FEASIBLE twin has; the stated
            # clock time is added to the goal it is about. v3.0 instead replaced goal 0's
            # reading, so feasible and infeasible QT4-3 labels differed in more than the clock
            # time, and the parser copied the infeasible template (with an invented clock time)
            # onto feasible requests (held-out seeds 55, 75).
            #   absolute_time_mismatch: "start at 6:14 PM ... pause when X finishes" -> the start (goal 0)
            #   delay_time_mismatch:    "start 14 min after X finishes, at 11:40"  -> the delayed start
            #   impossible_early_end:   "finish together before 2:35 PM"           -> the finish
            if conflict_type == "absolute_time_mismatch":
                target = goals[0]
            else:
                target = next((g for g in goals if g.readings and g.readings[0].anchor.kind == "device_event"), goals[0])
            clock = temporal_conflict["conflict_time"]
            target.readings.append(
                TimeReading(
                    relation="at",
                    anchor=Anchor(kind="now"),
                    absolute_time=clock.split(" ")[-1],
                    tolerance_minutes=target.readings[0].tolerance_minutes,
                )
            )
        elif goals:
            goals[0] = _inject_conflicting_reading(
                goals[0], temporal_conflict, for_training, raw_goals[0].get("anchor")
            )

    return IntentIR(
        query=episode["query"],
        query_type=meta.get("query_type", "unknown"),
        case=case,
        base_time=base_time,
        goals=goals,
        conflict_type=conflict_type,
    )


def extract_pair(episode: dict[str, Any]) -> dict[str, Any]:
    """Return {"query", "ir_for_training", "debug_ground_truth", "meta"} for one episode."""
    training_ir = extract_episode(episode, for_training=True)
    debug_ir = extract_episode(episode, for_training=False)
    return {
        "query": episode["query"],
        "ir_for_training": compact_ir_dict(training_ir.to_dict()),
        "debug_ground_truth": debug_ir.to_dict(),
        "meta": episode["meta"],
    }


def extract_dataset(benchmark_dir: str, pattern: str = "qt4-*_seed_*.json") -> list[dict[str, Any]]:
    """Extract (query, IR) pairs for every QT4 episode file in `benchmark_dir`."""
    # Benchmark files sit directly in the dir; generator output is <run>/episodes/*.json.
    paths = sorted(
        glob.glob(os.path.join(benchmark_dir, "qt4-*.json"))
        + glob.glob(os.path.join(benchmark_dir, "*", "episodes", "qt4-*.json"))
    )
    pairs = []
    errors = []
    for path in paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                episode = json.load(f)
            pair = extract_pair(episode)
            pair["source_file"] = os.path.basename(path)
            pairs.append(pair)
        except Exception as e:  # keep going; report at the end
            errors.append((path, str(e)))
    if errors:
        print(f"[extract_dataset] {len(errors)} file(s) failed to extract:")
        for path, err in errors[:20]:
            print(f"  {os.path.basename(path)}: {err}")
    return pairs


if __name__ == "__main__":
    import sys

    bench_dir = sys.argv[1] if len(sys.argv) > 1 else "SimuHome/data/benchmark"
    out_path = sys.argv[2] if len(sys.argv) > 2 else "data/qt4_ir_pairs.jsonl"
    pairs = extract_dataset(bench_dir)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    print(f"Extracted {len(pairs)} pairs -> {out_path}")


In [ ]:
%%writefile irlib/grounding.py
"""Grounding filter: drop clock times the parser wrote that the user never said.

The v3 parser sometimes adds an absolute clock reading that is not in the request
(e.g. "at 16:00" for "start the dishwasher now"). Z3 then finds a genuine clash
with the other reading and calls a POSSIBLE request impossible. A clock time is
something the user states explicitly, so it can be checked against the sentence
deterministically: if no time in the text matches it, the reading is removed
before Z3 sees it. Relative offsets are not filtered; a stated time plus/minus an offset in the same
IR also counts as stated (see ground_ir).

Times are recognised in digit form ("12:23 PM", "1:39 pm", "12 36 PM", "7 PM") and
word form ("six fifty six PM", "twelve thirty six PM", "seven o'clock").
"""

from __future__ import annotations

import copy
import re

_UNITS = {
    "zero": 0, "oh": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
    "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50}


def _with_meridiem(h: int, m: int, mer: str | None) -> set[tuple[int, int]]:
    if not (0 <= h <= 23 and 0 <= m <= 59):
        return set()
    if mer:
        mer = mer.replace(".", "").lower()
        if mer == "pm" and h < 12:
            h += 12
        elif mer == "am" and h == 12:
            h = 0
        return {(h, m)}
    out = {(h, m)}
    if h < 12:
        out.add((h + 12, m))  # no AM/PM given: either reading
    return out


def _words_to_numbers(text: str) -> str:
    """'six fifty six pm' -> '6 56 pm'; 'twelve thirty six' -> '12 36'."""
    tokens = re.findall(r"[a-z]+|\d+|[^a-z\d\s]", text.lower())
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if t in _TENS:
            val = _TENS[t]
            if i + 1 < len(tokens) and tokens[i + 1] in _UNITS and 0 < _UNITS[tokens[i + 1]] < 10:
                val += _UNITS[tokens[i + 1]]
                i += 1
            out.append(str(val))
        elif t in _UNITS:
            val = _UNITS[t]
            if t == "oh" and i + 1 < len(tokens) and tokens[i + 1] in _UNITS:  # "ten oh three" -> "10 03"
                out.append(f"{_UNITS[tokens[i + 1]]:02d}")
                i += 2
                continue
            out.append(str(val))
        else:
            out.append(t)
        i += 1
    return " ".join(out)


def times_in_text(text: str) -> set[tuple[int, int]]:
    """Every (hour 0-23, minute) the text could be stating as a clock time."""
    found: set[tuple[int, int]] = set()
    for variant in (text.lower(), _words_to_numbers(text)):
        for h, m, mer in re.findall(r"\b(\d{1,2})\s*[:.\s]\s*(\d{2})\s*(a\.?m\.?|p\.?m\.?)?", variant):
            found |= _with_meridiem(int(h), int(m), mer or None)
        for h, mer in re.findall(r"\b(\d{1,2})\s*(a\.?m\.?|p\.?m\.?)", variant):
            found |= _with_meridiem(int(h), 0, mer)
        for h in re.findall(r"\b(\d{1,2})\s*o\s*'?\s*clock", variant):
            found |= _with_meridiem(int(h), 0, None)
    return found


def _closest_device(name: str, known: list[str], cutoff: float) -> str | None:
    """Closest real device id, preferring the same room (the id prefix before the device type)."""
    import difflib

    same_room = [k for k in known if k.split("_")[0] == name.split("_")[0]]
    for pool in (same_room, known):
        hit = difflib.get_close_matches(name, pool, n=1, cutoff=cutoff)
        if hit:
            return hit[0]
    return None


def repair_device_ids(compact: dict, known_ids: list[str], cutoff: float = 0.85) -> tuple[dict, int, int]:
    """Copy of `compact` whose device ids that do not exist in the house are replaced by the closest real
    one (e.g. the parser's 'utility_laundry_dryer_1' -> 'utility_room_laundry_dryer_1'). Uses only the
    house's device list, never the answer key. Returns (ir, n_repaired, n_unresolved)."""
    known = list(known_ids)
    known_set = set(known)
    out = copy.deepcopy(compact)
    repaired = unresolved = 0

    def fix(obj: dict) -> None:
        nonlocal repaired, unresolved
        dev = obj.get("device_id")
        if not dev or dev in known_set:
            return
        hit = _closest_device(dev, known, cutoff)
        if hit:
            obj["device_id"] = hit
            repaired += 1
        else:
            unresolved += 1

    for g in out.get("goals", []):
        for r in g.get("readings", []):
            if (r.get("anchor") or {}).get("kind") == "device_event":
                fix(r["anchor"])
        for t in g.get("targets", []):
            fix(t)
    return out, repaired, unresolved


def ground_ir(compact: dict, query: str) -> tuple[dict, int]:
    """Copy of `compact` without clock readings the query does not state. Returns (ir, n_dropped).

    A clock time counts as stated if it appears in the text, or if it is a stated time
    plus/minus an offset the same IR uses ("finishes around 4:42 PM ... start 19 minutes
    after" -> 17:01): the benchmark's own labels express such derived times this way.
    """
    stated = times_in_text(query)
    offsets = {int(round(r["offset_minutes"])) for g in compact.get("goals", []) for r in g.get("readings", [])
               if isinstance(r.get("offset_minutes"), (int, float))}
    derived = set()
    for h, m in stated:
        for o in offsets:
            for t in (h * 60 + m + o, h * 60 + m - o):
                derived.add(((t // 60) % 24, t % 60))
    stated = stated | derived
    out = copy.deepcopy(compact)
    dropped = 0
    for g in out.get("goals", []):
        kept = []
        for r in g.get("readings", []):
            at = r.get("absolute_time")
            if at:
                try:
                    h, m = (int(x) for x in at.split(":")[:2])
                except ValueError:
                    dropped += 1
                    continue
                if (h, m) not in stated:
                    dropped += 1
                    continue
            kept.append(r)
        g["readings"] = kept
    return out, dropped



_PREVIOUS_ACTION = re.compile(r"\bafter (the|that) previous\b", re.I)


def repair_split_chain(compact: dict, query: str) -> tuple[dict, int]:
    """Merge a "N minutes after the previous action" that the parser split into two readings.

    Seen in E1 (QT4-1 feasible seed 25): "... cool the fridge 8 minutes after the previous action" came out as
    two readings of one goal, `now + 8` AND `previous goal + 0` (another fold's parser: `+ 1`). Both cannot
    hold (the previous goal is at now + 6), so Z3 called a POSSIBLE request impossible or computed a wrong
    window. The fix applies only to exactly that shape: a goal with ONE reading `after <goal> + M` and ONE
    relative reading `after now + N` (N > 0, no clock time), in a request that says "after the previous ...".
    It becomes `after <goal> + N` (N is the stated "N minutes after the previous action"). The benchmark's labels never
    have this shape (checked over all 300 QT4 episodes: a goal with both a `now` and a goal reading occurs only
    in QT4-3, as a stated CLOCK time, which this leaves alone), so no real contradiction can be removed.
    Returns (repaired copy, number of goals repaired)."""
    if not _PREVIOUS_ACTION.search(query or ""):
        return compact, 0
    out = copy.deepcopy(compact)
    fixed = 0
    for g in out.get("goals", []):
        rs = g.get("readings") or []
        chain = [r for r in rs if (r.get("anchor") or {}).get("kind") == "goal" and r.get("relation") == "after"]
        rel_now = [r for r in rs if (r.get("anchor") or {}).get("kind") == "now" and r.get("relation") == "after"
                   and not r.get("absolute_time") and float(r.get("offset_minutes") or 0) > 0]
        if len(chain) == 1 and len(rel_now) == 1:
            chain[0]["offset_minutes"] = float(rel_now[0]["offset_minutes"])
            g["readings"] = [r for r in rs if r is not rel_now[0]]
            fixed += 1
    return out, fixed


def _devices(goal: dict) -> set:
    return {t.get("device_id") for t in goal.get("targets") or [] if t.get("device_id")}


def repair_chain_anchor(compact: dict) -> tuple[dict, int]:
    """Re-point "after the previous action" to the previous action ON THE SAME DEVICE.

    In SimuHome's requests "... set the dimmer to 60 in 15 minutes and bump IT to 100 23 minutes after the
    previous action" chains to the dimmer's own earlier action: in the benchmark's labels every chained QT4-1
    goal (56 of 56) refers to the latest earlier goal on the same device. The 5-fold parsers sometimes point the
    chain at the previous SENTENCE's device instead (e.g. the dehumidifier step chained to the air purifier),
    which puts the goal's window at a wrong time and would make the time check refuse a correct schedule. A
    goal-anchored "after" reading whose referenced goal shares no device with this goal is re-pointed to the
    latest earlier goal (in the IR's order) that does; nothing changes if there is none. Run after device-name
    repair so that typos do not hide a shared device. Returns (repaired copy, readings re-pointed)."""
    out = copy.deepcopy(compact)
    goals = out.get("goals", [])
    fixed = 0
    for i, g in enumerate(goals):
        mine = _devices(g)
        for r in g.get("readings") or []:
            a = r.get("anchor") or {}
            if a.get("kind") != "goal" or r.get("relation") != "after":
                continue
            ref = a.get("goal_ref")
            if not isinstance(ref, int) or not (0 <= ref < len(goals)) or mine & _devices(goals[ref]):
                continue
            same = [j for j in range(i) if j != ref and mine & _devices(goals[j])]
            if same:
                a["goal_ref"] = same[-1]
                fixed += 1
    return out, fixed


_CLOCK_SPAN = re.compile(r"\b(\d{1,2})\s*[:.\s]\s*(\d{2})\s*(am|pm)\b")


def _op_state_values(goal: dict) -> set:
    return {a.get("value") for t in goal.get("targets") or [] for a in t.get("asserts") or []
            if str(a.get("attribute", "")).endswith("OperationalState.OperationalState")}


def restore_stated_clock(compact: dict, query: str) -> tuple[dict, int]:
    """Put back a clock time the user STATED but the parser left out (the counterpart of ground_ir).

    In QT4-3 ("start the dryer at 6:56 PM ... pause it when dryer 2 finishes") the stated clock time is what
    makes a request impossible; the 5-fold parsers often drop it and copy the request's consistent twin, so
    Z3 sees no contradiction (17 of 36 in-scope impossible QT4-3 requests). This applies only to that shape
    (a "by <goal>" deadline and a device-finish anchor, and no clock reading at all) and only to times with
    AM/PM, placed as the benchmark's labels place them:
      "start ... at T"  -> the goal that starts a machine (OperationalState 1);
      "... by T"        -> the goal anchored on the other device's finish.
    Other mentions ("I expect it to finish around T", "my call starts at T") are left alone. None of the 50
    POSSIBLE QT4-3 requests states a clock time, so this cannot turn a possible request into an impossible
    one there. Returns (copy, readings added)."""
    goals = compact.get("goals") or []
    readings = [r for g in goals for r in g.get("readings") or []]
    has_deadline = any(r.get("relation") == "by" and (r.get("anchor") or {}).get("kind") == "goal" for r in readings)
    has_device = any((r.get("anchor") or {}).get("kind") == "device_event" for r in readings)
    if not (has_deadline and has_device) or any(r.get("absolute_time") for r in readings):
        return compact, 0
    text = _words_to_numbers(query or "").replace("p . m .", "pm").replace("a . m .", "am")
    out = copy.deepcopy(compact)
    added = 0
    for m in _CLOCK_SPAN.finditer(text):
        h, mi, mer = int(m.group(1)), int(m.group(2)), m.group(3)
        if not (1 <= h <= 12 and 0 <= mi <= 59):
            continue
        h = (h % 12) + (12 if mer == "pm" else 0)
        before = text[:m.start()]
        clause = re.split(r"[.;,!?]| so | and ", before)[-1]
        if re.search(r"\bat\s*$", clause) and re.search(r"\bstart\b", clause):
            target = [g for g in out["goals"] if 1 in _op_state_values(g)]
        elif re.search(r"\bby\s*$", clause):
            target = [g for g in out["goals"] if any((r.get("anchor") or {}).get("kind") == "device_event"
                                                     for r in g.get("readings") or [])]
        else:
            continue
        if not target or any(r.get("absolute_time") for r in target[0].get("readings") or []):
            continue
        target[0]["readings"].append({"relation": "at", "anchor": {"kind": "now"}, "absolute_time": f"{h:02d}:{mi:02d}"})
        added += 1
    return out, added


In [ ]:
%%writefile irlib/parse_eval.py
"""Score a parse cache the way the gate uses it (shared by the rewording-robustness notebook and the tests).

For every benchmark request: the gate's preprocessing (clock-time grounding, split-chain repair, stated-clock
restoration, device-name repair against the episode's real devices, chain-anchor repair), device finish times
from the gold labels (at runtime they come from the simulator), then Z3. `text_for` maps a benchmark query to the
text the parser actually saw (e.g. a reworded version); grounding is applied to THAT text, as at runtime.

Counted per (sub-type, case):
  verdict ok            possible judged possible / contradiction proven
  goals ok / refused    a correct schedule for a benchmark goal passes / would be refused by the time check
  goals unchecked       no predicted goal produces the goal's values, so the gate does not time-check it
  invalid               no usable parse (the gate lets the episode through unchecked)
QT4-3 "still running before X finishes" goals are state checks, not schedule times, and are not counted.
"""

from __future__ import annotations

import collections
import copy
import json
import os
import re
from typing import Callable, Optional

from .constraint_encoder import check_feasibility, goal_windows
from .grounding import ground_ir, repair_chain_anchor, repair_device_ids, repair_split_chain, restore_stated_clock
from .ir_schema import ir_from_compact


def gate_view(pred: dict, text: str, known_devices: list[str]) -> dict:
    """The parse after the gate's deterministic preprocessing (same order as EpisodeGate.ensure_verdict)."""
    pred = copy.deepcopy(pred)
    pred, _ = ground_ir(pred, text)
    pred, _ = repair_split_chain(pred, text)
    pred, _ = restore_stated_clock(pred, text)
    if known_devices:
        pred, _, _ = repair_device_ids(pred, known_devices)
    pred, _ = repair_chain_anchor(pred)
    return pred


def score(cache: dict, rows: list[dict], bench_dir: str, text_for: Optional[Callable[[str], Optional[str]]] = None):
    """Returns (stats by (query_type, case), list of problems). `cache` maps the text the parser saw -> {'ir': ...}."""
    text_for = text_for or (lambda q: q)
    stats = collections.defaultdict(collections.Counter)
    problems = []
    for row in rows:
        qt, case = row["meta"]["query_type"], row["meta"]["case"]
        gold = row["debug_ground_truth"]
        if gold["conflict_type"] == "completion_vs_pause":
            continue
        text = text_for(row["query"])
        if text is None:
            continue  # no valid rewording for this request: excluded, counted by the caller
        st = stats[(qt, case)]
        st["n"] += 1
        entry = cache.get(text)
        if not entry or "ir" not in entry:
            st["invalid"] += 1
            problems.append((row["source_file"], "invalid parse"))
            continue
        ep = json.load(open(os.path.join(bench_dir, row["source_file"]), encoding="utf-8"))
        known = sorted(set(re.findall(r'"device_id":\s*"([^"]+)"', json.dumps(ep["initial_home_config"]))))
        try:
            pred = gate_view(entry["ir"], text, known)
            finish = {r["anchor"]["device_id"]: r.get("resolved_at_minutes") for g in gold["goals"] for r in g["readings"]
                      if r["anchor"].get("kind") == "device_event"}
            for g in pred["goals"]:
                for r in g["readings"]:
                    if (r.get("anchor") or {}).get("kind") == "device_event":
                        r["resolved_at_minutes"] = finish.get(r["anchor"].get("device_id"))
            ir = ir_from_compact(pred, text, gold["base_time"])
            verdict = check_feasibility(ir).feasible
        except Exception as e:
            st["invalid"] += 1
            problems.append((row["source_file"], f"bad structure: {type(e).__name__}"))
            continue
        want = case == "feasible"
        st["verdict_ok"] += verdict == want
        if verdict != want:
            problems.append((row["source_file"], "wrong verdict: predicted " + ("possible" if verdict else "impossible")))
        if not (want and verdict):
            continue
        w = goal_windows(ir)
        tick = float(ep["initial_home_config"]["tick_interval"])
        for gi, g in enumerate(ep["eval"]["goals"]):
            if g["when"].get("relation") == "before" and int(g["when"]["tolerance_ticks"]) == 0:
                continue
            st["goals"] += 1
            t = g["when"]["at_tick"] * tick / 60.0
            want_vals = {(x["device_id"], a["attribute"], json.dumps(a["value"])) for x in g["targets"] for a in x["asserts"]
                         if not (a["attribute"].endswith("OnOff.OnOff") and len(x["asserts"]) > 1)}
            devs = {x["device_id"] for x in g["targets"]}
            cands = [pg for pg in ir.goals if any(tt.device_id in devs for tt in pg.targets)
                     and any((tt.device_id, a.attribute, json.dumps(a.value)) in want_vals for tt in pg.targets for a in tt.asserts)]
            if not cands:
                st["goals_unchecked"] += 1
            elif any(w.get(pg.goal_id) is None or w[pg.goal_id].contains(t, 0.25) for pg in cands):
                st["goals_ok"] += 1
            else:
                st["goals_refused"] += 1
                problems.append((row["source_file"], f"goal {gi}: a correct schedule would be refused"))
    return stats, problems


def table(stats: dict) -> list[dict]:
    """One row per sub-type: possible judged possible, contradictions proven, goals refused, invalid."""
    out = []
    for qt in sorted({k[0] for k in stats}):
        f, i = stats.get((qt, "feasible"), collections.Counter()), stats.get((qt, "infeasible"), collections.Counter())
        out.append({"qt": qt, "possible_ok": f"{f['verdict_ok']}/{f['n']}", "contradictions_proven": f"{i['verdict_ok']}/{i['n']}",
                    "goals_refused": f"{f['goals_refused']}/{f['goals']}", "goals_unchecked": f["goals_unchecked"],
                    "invalid": f["invalid"] + i["invalid"]})
    return out


In [ ]:
%%writefile irlib/rewording.py
"""Meaning-preserving rewording of benchmark requests, for the parser's robustness test.

An LLM rewrites a request in casual, varied English. Only the WORDING may change, so the benchmark's answer key
(goals, times, contradictions) still applies. `check` rejects a rewording that changed the meaning:
  - every number must survive (minutes, percentages, temperatures, clock times; "twenty" and "20" are the same,
    "80.0%" and "80%" are the same);
  - every room named in the original must still be named;
  - every device phrase "<device word> <number>" must survive (the device's identity).
Invalid rewordings are retried; the notebook reports how many requests never got a valid one.
"""

from __future__ import annotations

import collections
import re

from .grounding import _words_to_numbers

SYSTEM = (
    "You rewrite smart-home voice requests the way a real person would casually say them. Keep the meaning EXACTLY: "
    "the same devices (write every device exactly as given, e.g. 'living room light 1', 'dryer 2 in the utility "
    "room'), the same rooms, the same actions and settings, and every number and every clock time. You may change "
    "word order, sentence structure, filler words and the way times are expressed (e.g. '20 minutes from now' -> "
    "'in 20 minutes'; '8 minutes after the previous action' -> '8 minutes after that'; '6 14 PM' -> '6:14 pm'). "
    "Do not add, drop or merge requests. Do not fix or point out contradictions. Output only the rewritten request."
)

ROOMS = ("living room", "kitchen", "bedroom", "bathroom", "utility room", "office", "dining room", "kids room",
         "garage", "hallway", "study", "laundry room", "guest room")

_DEVICE_WORDS = r"(light|lamp|blinds?|fan|purifier|dehumidifier|humidifier|heat pump|air conditioner|ac|refrigerator|" \
                r"fridge|freezer|washer|dryer|dishwasher|vacuum|tv|television|dimmer|window covering|oven|plug|switch)"


def _normal(text: str) -> str:
    # number words -> digits ("twenty" -> "20"), then re-join decimals the tokenizer split ("80 . 0" -> "80.0")
    return re.sub(r"(\d) \. (\d)", lambda m: m.group(1) + "." + m.group(2), _words_to_numbers(text.lower()))


def _numbers(text: str) -> collections.Counter:
    t = _normal(text)
    nums = re.findall(r"\d+(?:\.\d+)?", t)
    return collections.Counter(f"{float(n):g}" for n in nums)   # "80.0" and "80" are the same number


def _devices(text: str) -> collections.Counter:
    t = _normal(text)
    return collections.Counter(f"{m.group(1)} {m.group(2)}" for m in re.finditer(_DEVICE_WORDS + r"\s+(\d+)\b", t))


def check(original: str, reworded: str) -> list[str]:
    """Reasons the rewording is not meaning-preserving ([] = accepted)."""
    why = []
    if not reworded or len(reworded) < 0.4 * len(original):
        return ["empty or much shorter"]
    a, b = _numbers(original), _numbers(reworded)
    if a != b:
        why.append(f"numbers differ: lost {dict(a - b)}, added {dict(b - a)}")
    lo, lr = original.lower(), reworded.lower()
    lost_rooms = [r for r in ROOMS if r in lo and r not in lr]
    if lost_rooms:
        why.append(f"rooms lost: {lost_rooms}")
    lost_dev = _devices(original) - _devices(reworded)
    if lost_dev:
        why.append(f"device phrases lost: {dict(lost_dev)}")
    if lr.strip() == lo.strip():
        why.append("unchanged")
    return why


## 2. Settings and inputs


In [ ]:
# ============================ EDIT THIS BLOCK ONLY ============================
TRAIN_REWRITER = 'llama3.1:8b'      # rewordings used as extra TRAINING data
TEST_REWRITER = 'mistral-nemo:12b'  # rewordings used ONLY for testing (a different model family)
ATTEMPTS = 3                        # rewordings tried per request until one passes the meaning check
BASE_MODEL = 'Qwen/Qwen3-1.7B'
EPOCHS, LR, LORA_R = 5, 2e-4, 16    # same recipe as the current parser
BATCH = 16                          # requests parsed at once (batched generation)
# ==============================================================================
from irlib.extract_ir_from_episode import extract_dataset
from irlib.rewording import SYSTEM as REWORD_SYSTEM, check
from irlib.parse_eval import score, table, gate_view
from irlib.constraint_encoder import check_feasibility, goal_windows
from irlib.ir_schema import ir_from_compact
native = extract_dataset(BENCH)
assert len(native) == 300
def find_all(pattern):
    return sorted(set(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) + glob.glob(f'**/{pattern}', recursive=True)))
# old parser: cache + adapters (folders, or inside an attached zip)
cache_path = next((p for p in find_all('parse_cache_v3_cv.json') if 'reworded' not in p), None)
assert cache_path, 'attach parser_v3_cv_outputs (needs parse_cache_v3_cv.json)'
OLD_CACHE = {k: v for k, v in json.load(open(cache_path, encoding='utf-8')).items() if k != '_meta'}
def scan_adapters(prefix='qwen3_ir_parser_adapter_v3_fold'):
    found = {}
    for cfg in find_all('adapter_config.json'):
        folder = os.path.dirname(cfg)
        m = re.fullmatch(re.escape(prefix) + r'(\d)', os.path.basename(folder))
        if m and os.path.exists(os.path.join(folder, 'adapter_model.safetensors')):
            found.setdefault(int(m.group(1)), folder)
    return found
OLD_ADAPTERS = scan_adapters()
if sorted(OLD_ADAPTERS) != [0, 1, 2, 3, 4]:
    for z in find_all('*.zip'):
        try:
            with zipfile.ZipFile(z) as zf:
                names = [n for n in zf.namelist() if 'qwen3_ir_parser_adapter_v3_fold' in n]
                if names:
                    print('unpacking old adapters from', z); zf.extractall('old_adapters', members=names)
        except zipfile.BadZipFile:
            pass
    OLD_ADAPTERS = scan_adapters()
print('old adapters:', {k: v for k, v in sorted(OLD_ADAPTERS.items())})
assert sorted(OLD_ADAPTERS) == [0, 1, 2, 3, 4], 'attach the 5 old fold adapters (parser_v3_cv_outputs)'
QWEN_REWORDINGS = {}
for p in find_all('rewordings.jsonl'):
    if p.startswith('/kaggle/input'):
        for line in open(p, encoding='utf-8'):
            d = json.loads(line)
            if d.get('ok'):
                QWEN_REWORDINGS[d['query']] = d['reworded']
print(f'{len(OLD_CACHE)} old parses; {len(QWEN_REWORDINGS)} Qwen2.5-14B rewordings attached (optional 2nd test set)')


## 3. Stage 1: reword all requests with the two models (meaning-checked)


In [ ]:
def reword_file(model):
    return f"rewordings_{model.replace(':', '_').replace('.', '_')}.jsonl"
def load_rewordings(model):
    out = {}
    if os.path.exists(reword_file(model)):
        for line in open(reword_file(model), encoding='utf-8'):
            d = json.loads(line); out[d['query']] = d
    return out
todo_models = [m for m in (TRAIN_REWRITER, TEST_REWRITER) if len(load_rewordings(m)) < len(native)]
if todo_models:
    if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
        !apt-get -qq install -y zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh
    env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NUM_PARALLEL='1', OLLAMA_MAX_LOADED_MODELS='1',
               OLLAMA_SCHED_SPREAD='1', OLLAMA_KEEP_ALIVE='30m')
    ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
        except Exception:
            time.sleep(1)
    def rewrite(model, q, attempt):
        body = {'model': model, 'temperature': 0.7 + 0.1 * attempt, 'seed': attempt,
                'messages': [{'role': 'system', 'content': REWORD_SYSTEM}, {'role': 'user', 'content': q}]}
        req = urllib.request.Request('http://127.0.0.1:11434/v1/chat/completions', method='POST',
                                     headers={'Content-Type': 'application/json'}, data=json.dumps(body).encode())
        return json.loads(urllib.request.urlopen(req, timeout=600).read())['choices'][0]['message']['content'].strip().strip('"').strip()
    for model in todo_models:
        subprocess.run(['ollama', 'pull', model], check=True)
        done = load_rewordings(model); t0 = time.time()
        with open(reword_file(model), 'a', encoding='utf-8') as log:
            for n, r in enumerate(native):
                q = r['query']
                if q in done:
                    continue
                for attempt in range(ATTEMPTS):
                    new = rewrite(model, q, attempt); why = check(q, new)
                    if not why:
                        break
                d = {'query': q, 'reworded': new, 'ok': not why, 'why': why, 'attempts': attempt + 1, 'model': model}
                log.write(json.dumps(d, ensure_ascii=False) + '\n'); log.flush(); done[q] = d
                if (n + 1) % 50 == 0:
                    print(f'  {model}: {n + 1}/{len(native)} ({time.time() - t0:.0f} s), valid {sum(x["ok"] for x in done.values())}', flush=True)
        subprocess.run(['ollama', 'stop', model])
    ollama_proc.terminate(); ollama_proc.wait(timeout=60)   # free the GPUs for training
TRAIN_RW = {q: d['reworded'] for q, d in load_rewordings(TRAIN_REWRITER).items() if d['ok']}
TEST_RW = {q: d['reworded'] for q, d in load_rewordings(TEST_REWRITER).items() if d['ok']}
print(f'valid rewordings: training ({TRAIN_REWRITER}) {len(TRAIN_RW)}/300, test ({TEST_REWRITER}) {len(TEST_RW)}/300')
for q in list(TEST_RW)[:3]:
    print('\nORIGINAL :', q[:200]); print('TRAIN RW :', TRAIN_RW.get(q, '-')[:200]); print('TEST RW  :', TEST_RW[q][:200])


## 4. Stage 2: retrain the 5 fold parsers (original + Llama rewordings of the TRAINING requests only)


In [ ]:
import torch, inspect
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
SYSTEM_PROMPT = (
    'You convert a smart-home scheduling request into a strict JSON '
    'intermediate representation (IR). Output ONLY the JSON object, '
    'no prose, no code fences.'
)
def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
def split(k):   # identical to notebook 01: same folds, same train/validation split
    held_out = [p for p in native if int(p['meta']['seed']) % 5 == k]
    trainable = [p for p in native if int(p['meta']['seed']) % 5 != k]
    random.Random(0).shuffle(trainable)
    n = int(len(trainable) * 0.9)
    return trainable[:n], trainable[n:], held_out
def augmented(rows):
    out = list(rows)
    for r in rows:
        if r['query'] in TRAIN_RW:
            out.append({**r, 'query': TRAIN_RW[r['query']]})   # same labels: the meaning did not change
    return out
def fmt(row):
    return {'text': f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{row['query']}\n<|assistant|>\n" + json.dumps(row['ir_for_training'], ensure_ascii=False)}
CC = torch.cuda.get_device_capability(0)
BF16 = CC[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float32
big_gpu = torch.cuda.get_device_properties(0).total_memory > 40e9
def run_tag(k):
    tr, va, te = split(k)
    assert not ({p['query'] for p in tr + va} & {p['query'] for p in te}), 'leak'
    labels = hashlib.sha256(json.dumps([[r['query'], r['ir_for_training']] for r in augmented(tr) + augmented(va)], sort_keys=True).encode()).hexdigest()[:12]
    return f'{BASE_MODEL}|fold={k}|aug={TRAIN_REWRITER}|data={labels}|epochs={EPOCHS}|lr={LR}|r={LORA_R}|best_by_eval_loss'
def done_tag(folder, tag):
    f = os.path.join(folder, 'TRAINED_WITH.txt')
    return os.path.exists(f) and open(f).read().strip() == tag
NEW_ADAPTERS = {}
for k in range(5):
    out, tag = f'qwen3_ir_parser_adapter_v3aug_fold{k}', run_tag(k)
    NEW_ADAPTERS[k] = out
    if os.path.isdir(out) and done_tag(out, tag):
        print('fold', k, 'already trained'); continue
    tr, va, te = split(k)
    tr_rows, va_rows = augmented(tr), augmented(va)
    # no request of fold k (original or reworded) may be in its training or validation data
    test_q = {r['query'] for r in te} | {TRAIN_RW.get(r['query']) for r in te} | {TEST_RW.get(r['query']) for r in te}
    assert not ({r['query'] for r in tr_rows + va_rows} & test_q), f'leak in fold {k}'
    print(f'TRAINING fold {k}: {len(tr_rows)} train ({len(tr)} original + {len(tr_rows) - len(tr)} reworded), {len(va_rows)} validation')
    ckpt = f'checkpoints_{out}'
    shutil.rmtree(ckpt, ignore_errors=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map={'': 0})
    lora = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
    len_kw = 'max_length' if 'max_length' in inspect.signature(SFTConfig.__init__).parameters else 'max_seq_length'
    bs, acc = (16, 1) if big_gpu else (2, 8)
    cfg = SFTConfig(output_dir=ckpt, per_device_train_batch_size=bs, gradient_accumulation_steps=acc, num_train_epochs=EPOCHS,
                    learning_rate=LR, logging_steps=10, save_strategy='epoch', save_total_limit=2, eval_strategy='epoch',
                    bf16=BF16, fp16=not BF16, report_to=[], dataset_text_field='text', loss_type='nll',
                    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, seed=k,
                    gradient_checkpointing=not big_gpu, **{len_kw: 1024})
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=Dataset.from_list([fmt(r) for r in tr_rows]),
                         eval_dataset=Dataset.from_list([fmt(r) for r in va_rows]), peft_config=lora)
    trainer.train()
    json.dump(trainer.state.log_history, open(f'train_log_{out}.json', 'w'), indent=2)
    trainer.model.save_pretrained(out)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(out)
    open(os.path.join(out, 'TRAINED_WITH.txt'), 'w').write(tag)
    del trainer, model; free_gpu(); shutil.rmtree(ckpt, ignore_errors=True)


## 5. Stage 3: parse with the OLD and the NEW parser (each request by its own fold, held-out only)


In [ ]:
def load_parser(adapter):
    tok = AutoTokenizer.from_pretrained(adapter)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map={'': 0})
    return tok, PeftModel.from_pretrained(base, adapter).eval()
def parse_many(tok, model, texts):
    out = {}
    for i in range(0, len(texts), BATCH):
        chunk = texts[i:i + BATCH]
        prompts = [f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{t}\n<|assistant|>\n" for t in chunk]
        enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=700, do_sample=False, pad_token_id=tok.pad_token_id)
        for t, g in zip(chunk, gen):
            raw = tok.decode(g[enc['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                out[t] = {'ir': json.loads(raw)}
            except Exception as e:
                out[t] = {'error': repr(e)[:200], 'raw': raw[:400]}
    return out
SETS = {'original': {r['query']: r['query'] for r in native}, 'test_reworded': TEST_RW}
if QWEN_REWORDINGS:
    SETS['qwen_reworded'] = QWEN_REWORDINGS
fold_of = {r['query']: int(r['meta']['seed']) % 5 for r in native}
CACHES = {}
for which, adapters in (('old', OLD_ADAPTERS), ('new', NEW_ADAPTERS)):
    for name in SETS:
        path = f'parse_cache_{which}_{name}.json'
        CACHES[(which, name)] = json.load(open(path, encoding='utf-8')) if os.path.exists(path) else {}
    for k in range(5):
        todo = {name: [t for q, t in texts.items() if fold_of[q] == k and t not in CACHES[(which, name)]] for name, texts in SETS.items()}
        if not any(todo.values()):
            continue
        tok, model = load_parser(adapters[k]); t0 = time.time()
        for name, texts in todo.items():
            got = parse_many(tok, model, texts)
            for t, v in got.items():
                v['fold'] = k
            CACHES[(which, name)].update(got)
            json.dump(CACHES[(which, name)], open(f'parse_cache_{which}_{name}.json', 'w', encoding='utf-8'), ensure_ascii=False)
        print(f'{which} parser, fold {k}: {sum(map(len, todo.values()))} requests parsed in {time.time() - t0:.0f} s', flush=True)
        del model; free_gpu()


## 6. Stage 4: results

Same requests and answer key for every column; reworded sets include only requests whose rewording passed the
meaning check. Then: how many **gate decisions** (possible/impossible verdict and time windows) on the ORIGINAL
requests differ between the old and the new parser. If (almost) none differ, the existing agent results carry over.


In [ ]:
rows_by_q = {r['query']: r for r in native}
report = {'train_rewriter': TRAIN_REWRITER, 'test_rewriter': TEST_REWRITER, 'n_train_rewordings': len(TRAIN_RW),
          'n_test_rewordings': len(TEST_RW), 'n_qwen_rewordings': len(QWEN_REWORDINGS), 'tables': {}}
def scored(which, name):
    texts = SETS[name]
    kept = [rows_by_q[q] for q in texts]
    st, problems = score(CACHES[(which, name)], kept, BENCH, text_for=lambda q: texts.get(q))
    return st, problems
def totals(st):
    f = [c for (qt, cs), c in st.items() if cs == 'feasible']; i = [c for (qt, cs), c in st.items() if cs == 'infeasible']
    return (f"{sum(c['verdict_ok'] for c in f)}/{sum(c['n'] for c in f)}", f"{sum(c['verdict_ok'] for c in i)}/{sum(c['n'] for c in i)}",
            f"{sum(c['goals_refused'] for c in f)}/{sum(c['goals'] for c in f)}")
print(f"{'test set':16s} {'parser':6s} | per sub-type: possible ok / contradictions proven / correct goals refused | totals")
for name in SETS:
    for which in ('old', 'new'):
        st, _ = scored(which, name)
        tb = table(st); report['tables'][f'{which}|{name}'] = {'by_subtype': tb, 'totals': totals(st)}
        cells = '  '.join(f"{t['qt']}: {t['possible_ok']} / {t['contradictions_proven']} / {t['goals_refused']}" for t in tb)
        print(f'{name:16s} {which:6s} | {cells} | {totals(st)}')
# gate decisions on the ORIGINAL requests: old (batched) vs new (batched); old cache (one-by-one) vs old batched = batching noise
def signature(entry, row):
    if not entry or 'ir' not in entry:
        return ('invalid',)
    gold = row['debug_ground_truth']
    ep = json.load(open(os.path.join(BENCH, row['source_file']), encoding='utf-8'))
    known = sorted(set(re.findall(r'"device_id":\s*"([^"]+)"', json.dumps(ep['initial_home_config']))))
    try:
        pred = gate_view(entry['ir'], row['query'], known)
        finish = {r['anchor']['device_id']: r.get('resolved_at_minutes') for g in gold['goals'] for r in g['readings'] if r['anchor'].get('kind') == 'device_event'}
        for g in pred['goals']:
            for r in g['readings']:
                if (r.get('anchor') or {}).get('kind') == 'device_event':
                    r['resolved_at_minutes'] = finish.get(r['anchor'].get('device_id'))
        ir = ir_from_compact(pred, row['query'], gold['base_time'])
        if not check_feasibility(ir).feasible:
            return ('impossible',)
        w = goal_windows(ir)
        goals = sorted((tuple(sorted((t.device_id, a.attribute, json.dumps(a.value)) for t in g.targets for a in t.asserts)),
                        None if w.get(g.goal_id) is None or w[g.goal_id].lo is None else round(w[g.goal_id].lo, 2),
                        None if w.get(g.goal_id) is None or w[g.goal_id].hi is None else round(w[g.goal_id].hi, 2)) for g in ir.goals)
        return ('possible', tuple(goals))
    except Exception as e:
        return ('error', type(e).__name__)
diff = collections.Counter(); noise = collections.Counter(); changed = []
for r in native:
    q = r['query']; cell = (r['meta']['query_type'], r['meta']['case'])
    s_old_b = signature(CACHES[('old', 'original')].get(q), r)
    s_new = signature(CACHES[('new', 'original')].get(q), r)
    s_old_1 = signature(OLD_CACHE.get(q), r)
    diff[cell] += s_new != s_old_1; noise[cell] += s_old_b != s_old_1
    if s_new != s_old_1:
        changed.append((r['source_file'], s_old_1[0], s_new[0]))
print('\nGate decisions on the 300 ORIGINAL requests that differ from the parser used in the agent runs:')
for cell in sorted(diff):
    print(f'  {cell[0]} {cell[1]:10s}: new parser {diff[cell]:2d}   (old parser re-run in batches: {noise[cell]} = batching noise)')
print(f'  TOTAL: {sum(diff.values())} of 300 differ with the new parser; {sum(noise.values())} differ just from re-parsing in batches')
print('  verdict flips (possible <-> impossible):', sum(a != b for _, a, b in changed))
report.update(changed_gate_decisions=changed, n_changed=sum(diff.values()), batching_noise=sum(noise.values()))
json.dump(report, open('reworded_parser_report.json', 'w'), indent=2)
# new parse cache in the format the experiment notebooks use (original requests, each by its own fold)
json.dump({'_meta': {'ir_version': 3, 'cv': True, 'augmented_with': TRAIN_REWRITER, 'created': time.strftime('%Y-%m-%d %H:%M')},
           **{q: v for q, v in CACHES[('new', 'original')].items()}}, open('parse_cache_v3aug_cv.json', 'w', encoding='utf-8'), ensure_ascii=False)
with zipfile.ZipFile(os.path.join(WORK, 'reworded_parser_results.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ['reworded_parser_report.json', 'parse_cache_v3aug_cv.json'] + glob.glob('rewordings_*.jsonl') + glob.glob('parse_cache_old_*.json') \
             + glob.glob('parse_cache_new_*.json') + glob.glob('train_log_qwen3_ir_parser_adapter_v3aug_fold*.json'):
        z.write(p)
with zipfile.ZipFile(os.path.join(WORK, 'reworded_parser_adapters.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    for k, d in NEW_ADAPTERS.items():
        for root, _, files in os.walk(d):
            for fn in files:
                z.write(os.path.join(root, fn))
print('\nwrote reworded_parser_results.zip and reworded_parser_adapters.zip')
